# 머신러닝 파일별 4 · 3-2 분류 평가지표 · 혼동행렬 · 임곗값 · ROC · PR

**한 줄 요약**

- 정확도 90%를 믿어도 되나?
- digits의 7 vs 나머지로 정확도의 함정 · 혼동행렬 · 정밀도 · 재현율 · F1
- validation에서 임곗값 고르기 · ROC-AUC · PR-AP까지

- **원본** · `6.머닝러신/03_전처리_평가지표_Titanic/3-2_평가지표_06_metrics.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-3 (평가지표 · 로지스틱 · 임곗값) 1 · 2 · 4 · 5장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 남자면 사망 규칙 0.777 · 전부 0 모델 0.900으로 정확도 하나의 함정을 설명한다
2. Positive를 먼저 정하고 혼동행렬 4칸 TN · FP · FN · TP를 읽는다
3. 정밀도 · 재현율 · F1을 칸 숫자로 직접 계산하고, Balanced Accuracy = Macro Recall을 보인다
4. 선택 규칙을 먼저 정하고 validation에서 임곗값을 고른 뒤 test를 1회만 평가한다
5. ROC-AUC와 PR-AP가 각각 무엇을 요약하는지 말한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">'정확도 95%' 보고를 받았을 때</td>
<td style="text-align: left;">찍기 기준선 · 혼동행렬부터 (1 · 2장)</td>
</tr>
<tr>
<td style="text-align: left;">스팸 필터 · 암 진단처럼 실수 비용이 다를 때</td>
<td style="text-align: left;">정밀도 vs 재현율 (2-2)</td>
</tr>
<tr>
<td style="text-align: left;">모델의 확률을 0 / 1로 바꿀 기준을 정할 때</td>
<td style="text-align: left;">validation에서 임곗값 (3장)</td>
</tr>
<tr>
<td style="text-align: left;">임곗값을 정하기 전에 모델끼리 비교할 때</td>
<td style="text-align: left;">ROC-AUC · PR-AP (4장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfm4_map.png" alt="3-2 평가지표 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 + 실습 1시간 (전체 실행 30초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">실행 환경 · 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">정확도의 함정</td>
<td style="text-align: left;">학습 없는 규칙이나 전부 0으로 찍는 모델도 왜 정확도가 높게 나오는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">혼동행렬과 지표</td>
<td style="text-align: left;">Positive를 정한 뒤 혼동행렬 4칸에서 정밀도 · 재현율 · F1은 어떻게 나오는가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">임곗값</td>
<td style="text-align: left;">확률을 0 / 1로 바꾸는 기준은 어떤 순서로 · 어느 데이터로 고르는가?</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">ROC · PR 곡선</td>
<td style="text-align: left;">임곗값 하나를 고르기 전, 모든 기준에서의 구분 능력은 어떻게 요약하는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">학습한 로지스틱은 '남자면 사망' 규칙을 어떤 지표에서 얼마나 넘는가?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

1-1과 종합 실습은 원본 폴더의 `train.csv`(타이타닉), 나머지는 scikit-learn 내장 digits를 씀. **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">numpy · pandas · scikit-learn (원본의 plotly 대신 정적 그림)</td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;">타이타닉 분할 0 · digits 분할 11 · 임곗값 실험 42 (원본과 같음)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;"><code>../03_전처리_평가지표_Titanic/train.csv</code> 891명 · digits 1,797장 (7 = Positive, 10%)</td>
</tr>
<tr>
<td style="text-align: left;">원본 그림</td>
<td style="text-align: left;">설명 스크린샷 9장 → 각 섹션의 💡 · 비교 표 · 장 그림으로</td>
</tr>
</tbody>
</table>

In [1]:
# 준비: 라이브러리 · 버전 (원본의 plotly 그림은 images/의 정적 그림으로 바꿈)
import numpy as np                                    # 배열
import pandas as pd                                   # 표
import sklearn                                        # 버전 확인
# 자주 쓰는 지표
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split  # 분할

print("sklearn:", sklearn.__version__)                          # 원본은 1.6.1
print("numpy  :", np.__version__, "· pandas:", pd.__version__)  # 배열 · 표

sklearn: 1.9.0
numpy  : 2.4.6 · pandas: 3.0.3


### 원본 목표 · 왜 정확도만 보면 안 되는가

- **핵심 질문** · 정확도 90%를 믿어도 되나? → digits의 **7 vs others**로 확인
- 목표 · 정확도의 함정 · 혼동행렬 · 정밀도 · 재현율 · F1 · 임곗값 · ROC-AUC · PR-AP
- 지난 회차의 train / validation / test 역할을 오늘 임곗값 선택에도 그대로 적용

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">상황</th>
<th style="text-align: left;">정확도</th>
<th style="text-align: left;">실제 성능</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">암 환자 1%, 건강한 사람 99%</td>
<td style="text-align: left;">"모두 건강" → 99%</td>
<td style="text-align: left;">암 환자를 <strong>전혀 못 찾음</strong></td>
</tr>
<tr>
<td style="text-align: left;">스팸 10%, 정상 90%</td>
<td style="text-align: left;">"모두 정상" → 90%</td>
<td style="text-align: left;">스팸을 <strong>전혀 못 걸러냄</strong></td>
</tr>
<tr>
<td style="text-align: left;">금융 사기 0.1%</td>
<td style="text-align: left;">"모두 정상" → 99.9%</td>
<td style="text-align: left;">사기를 <strong>전혀 못 잡음</strong></td>
</tr>
</tbody>
</table>

**원본 힌트**

- 모델을 믿기 전에, 우리가 만든 실험을 먼저 의심해야 함
- 한 번 나온 정확도를 너무 빨리 믿지 않고, 무엇을 맞추고 무엇을 틀렸는지 여러 근거로 확인

---
## 1. 정확도의 함정

> **이 장의 흐름**
>
> - 타이타닉에서 '남자면 사망' 규칙의 정확도 → digits 7 vs 나머지의 불균형 확인
> - 그다음 전부 0으로 찍는 가짜 모델의 정확도 · 정밀도 · 재현율

<img src="images/pfm4_illusion.png" alt="학습 없는 규칙 · 전부 0 모델의 정확도" width="642">

**파이프라인 위치** · ⑥ 평가 — 기준선

### 1-1. `A` 타이타닉 · '남자면 사망' 규칙 · `BaseEstimator` · `fillna` <sub>원본 셀 4 ~ 8</sub>

> 🎯 **학습 없이 '남자면 사망, 아니면 생존' 규칙만으로 정확도는 몇이 나오는가?**

💡 **쉽게 말하면**

- 이건 학습이 아니라 **규칙 한 줄**.
- 그런데도 77%가 나옴 → 정확도만 보면 착시.

**실행 흐름**

1. train.csv 8 : 2 분할
2. Age 결측은 train 평균으로 (3-1 회수)
3. 규칙 분류기로 test 정확도

**먼저 예측** · 규칙 한 줄의 정확도는 50% 근처일까? — 답은 코드 실행 결과 아래에

In [2]:
from sklearn.base import BaseEstimator  # fit · predict 틀만 빌림

CSV = "../03_전처리_평가지표_Titanic/train.csv"                     # 원본 폴더의 Kaggle Titanic
titanic_df = pd.read_csv(CSV)                                       # 891명
train_df, test_df = train_test_split(titanic_df, test_size=0.2,     # 원본과 같은 분할
                                     stratify=titanic_df["Survived"], random_state=0)
age_fill_value = train_df["Age"].mean()  # 결측 대체값은 train에서만 (3-1 누수 회수)
print(f"Age 결측 대체값(train 평균만 사용): {age_fill_value:.4f}")  # 원본과 같은 값
# test는 transform만
X_test_titanic = test_df.drop("Survived", axis=1).fillna({"Age": age_fill_value})
y_test_titanic = test_df["Survived"]                                # 정답 (1 = 생존)


class MyDummyClassifier(BaseEstimator):  # 학습 없는 규칙
    def fit(self, X, y=None):            # 아무것도 배우지 않음
        return self                      # 그대로 돌려줌

    def predict(self, X):                                                 # 남자면 0, 아니면 1
        return np.where(X["Sex"].to_numpy() == "male", 0, 1).astype(int)  # 성별만 봄


pred_dummy = MyDummyClassifier().fit(None).predict(X_test_titanic)  # 규칙으로 예측
print("Dummy Classifier 정확도:", round(accuracy_score(y_test_titanic, pred_dummy), 4))  # 0.7765
print("test 생존 비율:", round(y_test_titanic.mean(), 4))           # 찍기 기준선 비교용

Age 결측 대체값(train 평균만 사용): 29.4698
Dummy Classifier 정확도: 0.7765
test 생존 비율: 0.3855


**결과** · 예측 정답은 아님 — 0.7765

- `Age 결측 대체값(train 평균만 사용): 29.4698` — train에서만 계산한 평균
- `Dummy Classifier 정확도: 0.7765` — 규칙 한 줄인데 77.65%
- `test 생존 비율: 0.3855` — 전부 사망으로 찍어도 0.6145 → 규칙은 그보다 높을 뿐

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">규칙 분류기 (Dummy)</td>
<td style="text-align: left;">학습 없이 한 줄 규칙</td>
<td style="text-align: left;">모델이 넘어야 할 기준선</td>
</tr>
<tr>
<td style="text-align: left;">다수 클래스로 찍기</td>
<td style="text-align: left;">가장 많은 클래스 하나로</td>
<td style="text-align: left;">가장 낮은 기준선</td>
</tr>
<tr>
<td style="text-align: left;">학습한 모델 (파일별 5)</td>
<td style="text-align: left;">여러 특성으로 학습</td>
<td style="text-align: left;">기준선보다 높아야 의미</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 오늘 Dummy는 Sex만 봐서 결측 대체값이 숫자를 바꾸지 않음 · 그래도 **fit 괄호 안을 보는 습관**은 지킴
> - 이 0.7765는 파일별 5(4-1)에서 학습한 로지스틱과 다시 비교함

> ✅ **한 줄 정리** · 학습 없는 규칙 한 줄도 정확도 0.7765가 나오므로, 정확도는 기준선과 함께 읽어야 한다.

**딥러닝 코드에서는**

- 첫 epoch 전 '전부 다수 클래스' 정확도를 먼저 계산해 두기

### 1-2. `A` digits · 7 vs 나머지 · `load_digits` · `value_counts(normalize=True)` <sub>원본 셀 9 ~ 12</sub>

> 🎯 **숫자 7만 Positive로 두면 클래스 비율은 어떻게 되는가?**

💡 **쉽게 말하면**

- 한 클래스가 **극소수**면 불균형 데이터.
- 7은 10%뿐 → '전부 아님'이라고만 해도 90%.

**실행 흐름**

1. digits 1,797장
2. 7이면 1, 아니면 0
3. 8 : 2 분할 (seed 11) · test 비율

**먼저 예측** · test에서 7의 비율은? — 답은 코드 실행 결과 아래에

In [3]:
from sklearn.datasets import load_digits  # 8 × 8 손글씨 숫자

digits = load_digits()                                                # 1797장
X_all_7 = digits.data                                                 # 픽셀 64개
y_all_7 = (digits.target == 7).astype(int)                            # 7이면 1, 아니면 0
X_train_7, X_test_7, y_train_7, y_test_7 = train_test_split(          # 원본과 같은 분할
    X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=11)
# 0.9 · 0.1
print("테스트셋 클래스 비율:", pd.Series(y_test_7).value_counts(normalize=True).round(3).to_dict())
print("test 장수:", len(y_test_7), "· 그중 7:", int(y_test_7.sum()))  # 360장 중 36장

테스트셋 클래스 비율: {0: 0.9, 1: 0.1}
test 장수: 360 · 그중 7: 36


**결과** · 예측 정답은 0.1 (360장 중 36장)

- `테스트셋 클래스 비율: {0: 0.9, 1: 0.1}` — 7은 10% → 불균형
- `test 장수: 360 · 그중 7: 36` — 원본 히스토그램의 두 막대 = 7이 아닌 것 · 7

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">암 진단</td>
<td style="text-align: left;">양성 1%</td>
<td style="text-align: left;">'모두 음성' → 99%</td>
</tr>
<tr>
<td style="text-align: left;">금융 사기</td>
<td style="text-align: left;">양성 0.1%</td>
<td style="text-align: left;">'모두 음성' → 99.9%</td>
</tr>
<tr>
<td style="text-align: left;">스팸 메일 · digits 7</td>
<td style="text-align: left;">양성 10%</td>
<td style="text-align: left;">'모두 음성' → 90%</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 불균형에서는 정확도가 쉽게 높아짐 → 지표를 여러 개 봄

> ✅ **한 줄 정리** · 7을 Positive로 두면 test 360장 중 36장(10%)뿐인 불균형 데이터가 된다.

**딥러닝 코드에서는**

- 불량 · 질병 · 사기 탐지 데이터셋은 대부분 이렇게 한쪽이 드묾

### 1-3. `A` 전부 0 모델 · `precision_score` · `recall_score` · `zero_division=0` <sub>원본 셀 13 ~ 15</sub>

> 🎯 **전부 0으로 찍는 모델의 정확도 · 정밀도 · 재현율은?**

💡 **쉽게 말하면**

- 7을 **하나도 못 찾았는데** 정확도는 90%.
- 정밀도 · 재현율이 그 사실을 바로 드러냄.

**실행 흐름**

1. 항상 0을 내는 가짜 분류기
2. 정확도 · 정밀도 · 재현율

**먼저 예측** · 정확도 0.9일 때 재현율은? — 답은 코드 실행 결과 아래에

In [4]:
class FakeClassifier(BaseEstimator):  # 항상 0
    def fit(self, X, y=None):         # 배우지 않음
        return self                   # 그대로

    def predict(self, X):                   # 전부 0
        return np.zeros(len(X), dtype=int)  # '7 아님'만


fakepred_7 = FakeClassifier().fit(X_train_7, y_train_7).predict(X_test_7)  # 가짜 예측
print(f"Accuracy: {accuracy_score(y_test_7, fakepred_7):.3f}")             # 전체 성적표
# 7이라고 한 것 중 진짜 7
print("Precision:", precision_score(y_test_7, fakepred_7, zero_division=0))
print("Recall   :", recall_score(y_test_7, fakepred_7, zero_division=0))   # 진짜 7 중 찾은 것

Accuracy: 0.900
Precision: 0.0
Recall   : 0.0


**결과** · 예측 정답은 0.0

- `Accuracy: 0.900` — 전부 0인데 90%
- `Precision: 0.0` — 7이라고 한 적이 없음 → 0으로 나누기라 `zero_division=0`
- `Recall   : 0.0` — 진짜 7 36장을 하나도 못 찾음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도</td>
<td style="text-align: left;">전체 중 맞춘 비율</td>
<td style="text-align: left;">클래스가 균형일 때</td>
</tr>
<tr>
<td style="text-align: left;">정밀도</td>
<td style="text-align: left;">양성 예측 중 실제 양성</td>
<td style="text-align: left;">오탐을 줄여야 할 때 (스팸)</td>
</tr>
<tr>
<td style="text-align: left;">재현율</td>
<td style="text-align: left;">실제 양성 중 찾은 비율</td>
<td style="text-align: left;">누락을 줄여야 할 때 (암)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `zero_division=0`이 없으면 양성 예측이 0개일 때 경고가 나옴 → 값을 0으로 정해 둠

> ✅ **한 줄 정리** · 전부 0 모델은 정확도 0.900이어도 정밀도 · 재현율이 0이며, 이것이 정확도의 함정이다.

**딥러닝 코드에서는**

- 불균형 학습에서 loss가 잘 내려가도 소수 클래스 재현율이 0일 수 있음 → 클래스별 지표 기록

---
## 2. 혼동행렬과 지표

> **이 장의 흐름**
>
> - Positive를 먼저 정하고 전부 0 모델의 혼동행렬 4칸 읽기
> - 그다음 로지스틱 기준선으로 정밀도 · 재현율 · F1을 칸 숫자로 계산 → 평균 방식 비교

<img src="images/pfm4_confusion.png" alt="기준선 로지스틱의 혼동행렬과 지표 공식" width="764">

**파이프라인 위치** · ⑥ 평가 — Positive 중심 성적표

### 2-1. `A` Positive 정하기 · 혼동행렬 · `confusion_matrix` · `ravel()` · `classification_report` <sub>원본 셀 16 ~ 20</sub>

> 🎯 **혼동행렬 4칸은 무엇이고, 전부 0 모델은 어느 칸이 가득 차는가?**

💡 **쉽게 말하면**

- **찾고 싶은 대상 = Positive** (digits는 7).
- 행 = 실제, 열 = 예측 → TN · FP / FN · TP.

**실행 흐름**

1. labels=[0, 1]로 혼동행렬
2. ravel()로 4칸 풀기
3. classification_report

**먼저 예측** · 전부 0 모델의 FN은 몇 칸일까? — 답은 코드 실행 결과 아래에

In [5]:
from sklearn.metrics import classification_report, confusion_matrix  # 4칸 · 클래스별 표

cm_7 = confusion_matrix(y_test_7, fakepred_7, labels=[0, 1])  # 행 = 실제, 열 = 예측
print("TN, FP, FN, TP:", *cm_7.ravel())                       # 원본 셀 19
print(classification_report(y_test_7, fakepred_7, digits=3, zero_division=0))  # 클래스별 지표

TN, FP, FN, TP: 324 0 36 0
              precision    recall  f1-score   support

           0      0.900     1.000     0.947       324
           1      0.000     0.000     0.000        36

    accuracy                          0.900       360
   macro avg      0.450     0.500     0.474       360
weighted avg      0.810     0.900     0.853       360



**결과** · 예측 정답은 36 — 진짜 7 전부

- `TN, FP, FN, TP: 324 0 36 0` — 7이 아닌 324장만 맞힘 · 7은 36장 전부 놓침(FN)
- `1      0.000     0.000     0.000        36` — Positive(7) 줄: 정밀도 · 재현율 · F1 모두 0
- `accuracy                          0.900       360` — 그래도 정확도는 0.900

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">TN · FP</td>
<td style="text-align: left;">실제 음성 → 맞춤 · 양성으로 틀림(<strong>오탐</strong>)</td>
<td style="text-align: left;">FP가 비싸면 정밀도</td>
</tr>
<tr>
<td style="text-align: left;">FN · TP</td>
<td style="text-align: left;">실제 양성 → 음성으로 틀림(<strong>누락</strong>) · 맞춤</td>
<td style="text-align: left;">FN이 비싸면 재현율</td>
</tr>
<tr>
<td style="text-align: left;">정확도 vs 나머지</td>
<td style="text-align: left;">전체 성적표 vs Positive 중심 성적표</td>
<td style="text-align: left;">Positive를 먼저 적고 시작</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - `labels=[0, 1]`이 없으면 예측에 1이 없을 때 2 × 2가 아닐 수 있음
> - Positive를 바꾸면(예: 7이 아닌 것) 지표 값이 전부 바뀜

> ✅ **한 줄 정리** · Positive를 먼저 정하면 혼동행렬 TN · FP · FN · TP로 어떤 실수를 했는지 바로 보인다.

**딥러닝 코드에서는**

- `torchmetrics.ConfusionMatrix` · 다중 분류는 10 × 10 혼동행렬로 헷갈리는 쌍 찾기

### 2-2. `A` 정밀도 · 재현율 · F1 직접 계산 · `TP / (TP + FP)` · `f1_score` <sub>원본 셀 21 ~ 25</sub>

> 🎯 **기준선 로지스틱의 4칸으로 정밀도 · 재현율 · F1을 손으로 계산하면?**

💡 **쉽게 말하면**

- 정밀도 = **헛발질 안 했나** · 재현율 = **놓치지 않았나**.
- F1은 둘의 조화평균 → 하나가 0이면 F1도 0 근처.

**실행 흐름**

1. 일부러 약하게 만들지 않은 로지스틱
2. 4칸 꺼내기
3. 공식으로 계산 · 함수와 비교

**먼저 예측** · FP가 0이면 정밀도는? — 답은 코드 실행 결과 아래에

In [6]:
from sklearn.linear_model import LogisticRegression  # 기준선 모델

lr_7 = LogisticRegression(max_iter=5000, random_state=42).fit(X_train_7, y_train_7)  # 원본 설정
y_pred_lr_7 = lr_7.predict(X_test_7)                                             # 기본 기준 0.5
tn, fp, fn, tp = confusion_matrix(y_test_7, y_pred_lr_7, labels=[0, 1]).ravel()  # 4칸
print(f"Logistic Accuracy: {accuracy_score(y_test_7, y_pred_lr_7):.4f}")         # 원본 셀 25
print("Baseline TN, FP, FN, TP:", tn, fp, fn, tp)                                # 칸 숫자
p_hand, r_hand = tp / (tp + fp), tp / (tp + fn)                                  # 공식 그대로
f1_hand = 2 * p_hand * r_hand / (p_hand + r_hand)                                # 조화평균
print(f"손 계산 P {p_hand:.4f} · R {r_hand:.4f} · F1 {f1_hand:.4f}")             # 직접
p_fn = precision_score(y_test_7, y_pred_lr_7)                                    # 함수로 정밀도
r_fn = recall_score(y_test_7, y_pred_lr_7)                                       # 함수로 재현율
f_fn = f1_score(y_test_7, y_pred_lr_7)                                           # 함수로 F1
print(f"함수   P {p_fn:.4f} · R {r_fn:.4f} · F1 {f_fn:.4f}")                     # sklearn과 같은지

Logistic Accuracy: 0.9944
Baseline TN, FP, FN, TP: 324 0 2 34
손 계산 P 1.0000 · R 0.9444 · F1 0.9714
함수   P 1.0000 · R 0.9444 · F1 0.9714


**결과** · 예측 정답은 1.000

- `Logistic Accuracy: 0.9944` — test 2장만 틀림
- `Baseline TN, FP, FN, TP: 324 0 2 34` — 오탐 0 · 7을 2장 놓침
- `손 계산 P 1.0000 · R 0.9444 · F1 0.9714` — 34 ÷ 34 = 1 · 34 ÷ 36 = 0.9444
- `함수   P 1.0000 · R 0.9444 · F1 0.9714` — 공식과 같은 값

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">스팸 메일 필터</td>
<td style="text-align: left;"><strong>정밀도</strong> 우선</td>
<td style="text-align: left;">정상 메일을 스팸으로 보내면 안 됨</td>
</tr>
<tr>
<td style="text-align: left;">암 진단 · 금융 사기</td>
<td style="text-align: left;"><strong>재현율</strong> 우선</td>
<td style="text-align: left;">놓치면 생명 · 큰 손실</td>
</tr>
<tr>
<td style="text-align: left;">추천 시스템</td>
<td style="text-align: left;">정밀도 · 재현율 · 랭킹(Top-K)</td>
<td style="text-align: left;">잘못된 추천은 이탈</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - '무엇을 틀리면 더 큰 문제인가?'를 먼저 생각하고 지표를 고름
> - F1은 하나가 낮으면 강하게 깎임 (균형을 요구)

> ✅ **한 줄 정리** · 정밀도는 TP ÷ (TP + FP), 재현율은 TP ÷ (TP + FN)이고, F1은 둘의 조화평균이다.

**딥러닝 코드에서는**

- 검출 모델 mAP · 분할 Dice도 결국 TP · FP · FN의 조합

### 2-3. `B` Balanced · Macro · Weighted · `balanced_accuracy_score` · `average=` <sub>원본 셀 26 · 27</sub>

> 🎯 **Balanced Accuracy와 Macro Recall은 왜 같고, Weighted는 무엇이 다른가?**

💡 **쉽게 말하면**

- Balanced = 클래스별 재현율의 **단순 평균** = Macro Recall (정의상 같음).
- Weighted는 표본 수로 가중 → 다수 클래스 쪽으로 기움.

**실행 흐름**

1. Balanced Accuracy
2. Macro · Weighted 정밀도 · 재현율 · F1
3. 클래스별 재현율 평균과 비교

**먼저 예측** · Balanced Accuracy는 정확도 0.9944보다 높을까 낮을까? — 답은 코드 실행 결과 아래에

In [7]:
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support  # 평균 방식

bal = balanced_accuracy_score(y_test_7, y_pred_lr_7)                       # 클래스별 재현율 평균
pm, rm, fm, _ = precision_recall_fscore_support(y_test_7, y_pred_lr_7, average="macro")  # 단순 평균
# 표본 수 가중
pw, rw, fw, _ = precision_recall_fscore_support(y_test_7, y_pred_lr_7, average="weighted")
print(f"Balanced Accuracy: {bal:.4f}")                                     # 원본 셀 26
print(f"Macro     P:{pm:.4f} R:{rm:.4f} F1:{fm:.4f}")                      # 소수 클래스도 같은 비중
print(f"Weighted  P:{pw:.4f} R:{rw:.4f} F1:{fw:.4f}")                      # 다수 클래스 쪽
r0, r1 = tn / (tn + fp), tp / (tp + fn)                                    # 클래스 0 · 1의 재현율
print(f"(재현율0 {r0:.4f} + 재현율1 {r1:.4f}) / 2 = {(r0 + r1) / 2:.4f}")  # Balanced와 같은지

Balanced Accuracy: 0.9722
Macro     P:0.9969 R:0.9722 F1:0.9842
Weighted  P:0.9945 R:0.9944 F1:0.9944
(재현율0 1.0000 + 재현율1 0.9444) / 2 = 0.9722


**결과** · 예측 정답은 낮음 — 0.9722

- `Balanced Accuracy: 0.9722` — 정확도 0.9944보다 낮음 → 7을 놓친 2장이 드러남
- `Macro     P:0.9969 R:0.9722 F1:0.9842` — Macro Recall = Balanced Accuracy
- `Weighted  P:0.9945 R:0.9944 F1:0.9944` — 다수 클래스(0) 쪽으로 기울어 정확도와 비슷
- `(재현율0 1.0000 + 재현율1 0.9444) / 2 = 0.9722` — 정의 그대로 계산해도 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Balanced Accuracy</td>
<td style="text-align: left;">클래스별 재현율 평균</td>
<td style="text-align: left;">클래스별 놓침을 같은 비중으로 볼 때</td>
</tr>
<tr>
<td style="text-align: left;">Macro</td>
<td style="text-align: left;">클래스별 지표의 단순 평균</td>
<td style="text-align: left;">소수 클래스도 같은 비중으로 볼 때</td>
</tr>
<tr>
<td style="text-align: left;">Weighted</td>
<td style="text-align: left;">표본 수를 반영한 평균</td>
<td style="text-align: left;">실제 분포 반영 · 다수 클래스 영향 확인</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 기준선 모델이 나쁘다는 뜻이 아님 → 정확도 한 숫자로는 어떤 클래스를 놓쳤는지 읽을 수 없다는 뜻
> - 읽는 순서 · 혼동행렬 → 7의 정밀도 · 재현율 → 클래스별 재현율 평균

> ✅ **한 줄 정리** · Balanced Accuracy는 클래스별 재현율의 평균이라 Macro Recall과 정의상 같고, 불균형에서 정확도보다 놓침을 잘 드러낸다.

**딥러닝 코드에서는**

- 다중 분류 리더보드의 macro-F1 · balanced accuracy — 소수 클래스를 놓치면 크게 깎임

---
## 3. 임곗값

> **이 장의 흐름**
>
> - 임곗값 실험 전용 train / validation / test 3분할 → 일부러 약한 모델(PCA 10 + C=0.1)
> - 규칙(F1 최대)을 먼저 정하고 validation 표에서 고름 → test 1회 → seed를 바꿔 절차 반복

<img src="images/pfm4_threshold.png" alt="validation에서 임곗값별 정밀도 · 재현율 · F1" width="590">

**파이프라인 위치** · ⑤ 선택(validation) → ⑥ 최종 평가(test 1회)

### 3-1. `A` 3분할과 일부러 약한 모델 · `PCA(n_components=10)` · `Pipeline` <sub>원본 셀 28 ~ 32</sub>

> 🎯 **임곗값을 고르려면 왜 validation이 따로 필요하고, 왜 약한 모델을 쓰는가?**

💡 **쉽게 말하면**

- 배울 때는 **train**, 고를 때는 **validation**, 마지막에 **test** 한 번.
- 완벽한 모델은 기준을 바꿔도 실수 교환이 안 보여 일부러 정보를 줄임.

**실행 흐름**

1. 전체 → 개발 80 · test 20
2. 개발 → train 75 · validation 25
3. PCA 10 + 로지스틱(C=0.1)을 train에만 fit
4. validation 확률

**먼저 예측** · train · validation · test는 각각 몇 장? — 답은 코드 실행 결과 아래에

In [8]:
from sklearn.decomposition import PCA  # 정보를 10개로 줄임
from sklearn.pipeline import Pipeline  # train에만 fit

X_dev, X_final_test_7, y_dev, y_final_test_7 = train_test_split(       # 3장 전용 분할 (seed 42)
    X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=42)
X_fit_7, X_val_7, y_fit_7, y_val_7 = train_test_split(                 # 개발 데이터를 다시 나눔
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print("train      :", X_fit_7.shape, "Positive=", int(y_fit_7.sum()))  # 학습용
print("validation :", X_val_7.shape, "Positive=", int(y_val_7.sum()))  # 기준 선택용
n_pos_test = int(y_final_test_7.sum())                                 # test의 7 개수
# 봉인
print("test       :", X_final_test_7.shape, "Positive=", n_pos_test, "(아직 선택에 사용하지 않음)")
weak_lr = LogisticRegression(max_iter=5000, C=0.1, random_state=42)    # 규제를 준 로지스틱
threshold_pipe_7 = Pipeline([("pca", PCA(n_components=10, random_state=42)),  # 일부러 약하게
                             ("logistic", weak_lr)])
threshold_pipe_7.fit(X_fit_7, y_fit_7)                                 # PCA까지 train에만 fit
proba_val_7 = threshold_pipe_7.predict_proba(X_val_7)[:, 1]            # 7일 확률
print("validation probability 개수:", len(proba_val_7))                # 360개

train      : (1077, 64) Positive= 107
validation : (360, 64) Positive= 36
test       : (360, 64) Positive= 36 (아직 선택에 사용하지 않음)


validation probability 개수: 360


**결과** · 예측 정답은 1077 · 360 · 360

- `train      : (1077, 64) Positive= 107` — 학습용 60%
- `validation : (360, 64) Positive= 36` — 기준을 고를 20%
- `test       : (360, 64) Positive= 36 (아직 선택에 사용하지 않음)` — 마지막 1회용 20%
- `validation probability 개수: 360` — 기준을 바꿔 볼 확률 360개

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">train</td>
<td style="text-align: left;">모델 · PCA를 학습</td>
<td style="text-align: left;">배울 때</td>
</tr>
<tr>
<td style="text-align: left;">validation</td>
<td style="text-align: left;">임곗값을 고름 (여러 번 봐도 됨)</td>
<td style="text-align: left;">선택할 때</td>
</tr>
<tr>
<td style="text-align: left;">test</td>
<td style="text-align: left;">모든 선택이 끝난 뒤 1회</td>
<td style="text-align: left;">보고할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 1 · 2장은 정해진 모델을 test에서 관찰만 했음 → 그 결과로 다시 고르지 않았으니 잘못된 절차가 아님
> - 선택에 쓴 데이터로 마지막 성능까지 재면 더 이상 처음 보는 시험이 아님

> ✅ **한 줄 정리** · 임곗값처럼 무언가를 고를 때는 validation이 필요하며, Pipeline 전체를 train에만 fit한다.

**딥러닝 코드에서는**

- `train / val / test` loader 3개 · val로 best.pt와 기준을 고르고 test는 마지막

### 3-2. `A` 규칙 먼저 · validation 표 · `proba >= thr` · `idxmax()` <sub>원본 셀 33 · 34</sub>

> 🎯 **임곗값을 낮추면 놓치는 Positive가 줄어드는 대가는 무엇이고, 어떤 규칙으로 고르는가?**

💡 **쉽게 말하면**

- 기준을 낮추면 **더 많이 7이라고 함** → 놓침(FN)은 줄고 헛발질(FP)은 늘 수 있음.
- 규칙(validation F1 최대)은 **표를 보기 전에** 정함.

**실행 흐름**

1. 규칙 선언: validation F1 최대
2. 기준 0.1 ~ 0.9마다 4개 지표
3. F1 최대 행 고르기

**먼저 예측** · 기준을 0.9 → 0.1로 낮추면 정밀도는? — 답은 코드 실행 결과 아래에

In [9]:
thresholds_7 = np.round(np.arange(0.1, 1.0, 0.1), 1)                  # 후보 9개
rows = []                                                             # 기준마다 지표
for thr in thresholds_7:                                              # 기준을 하나씩
    y_hat = (proba_val_7 >= thr).astype(int)                          # 확률 ≥ 기준 → 7
    rows.append({"threshold": thr,                                    # 네 지표
                 "precision": precision_score(y_val_7, y_hat, zero_division=0),
                 "recall": recall_score(y_val_7, y_hat), "f1": f1_score(y_val_7, y_hat),
                 "accuracy": accuracy_score(y_val_7, y_hat)})
thr_table_7 = pd.DataFrame(rows)                                      # 9행 표
print(thr_table_7.round(3).to_string(index=False))                    # 원본 셀 34의 표
best_row_7 = thr_table_7.loc[thr_table_7["f1"].idxmax()]              # 미리 선언한 규칙
selected_threshold_7 = float(best_row_7["threshold"])                 # 고른 기준
b = best_row_7                                                        # 짧은 이름
print(f"[validation F1 최대] threshold={selected_threshold_7:.1f}, "  # 원본 형식
      f"P={b['precision']:.3f}, R={b['recall']:.3f}, F1={b['f1']:.3f}")

 threshold  precision  recall    f1  accuracy
       0.1      0.854   0.972 0.909     0.981
       0.2      0.897   0.972 0.933     0.986
       0.3      0.921   0.972 0.946     0.989
       0.4      0.919   0.944 0.932     0.986
       0.5      0.943   0.917 0.930     0.986
       0.6      0.939   0.861 0.899     0.981
       0.7      0.968   0.833 0.896     0.981
       0.8      0.964   0.750 0.844     0.972
       0.9      1.000   0.694 0.820     0.969
[validation F1 최대] threshold=0.3, P=0.921, R=0.972, F1=0.946


**결과** · 예측 정답은 낮아짐 — 1.000 → 0.854

- `0.1      0.854   0.972 0.909     0.981` — 기준 0.1 · 많이 잡아 재현율 0.972 · 정밀도 0.854
- `0.9      1.000   0.694 0.820     0.969` — 기준 0.9 · 헛발질 0 · 대신 7을 30%나 놓침
- `[validation F1 최대] threshold=0.3, P=0.921, R=0.972, F1=0.946` — 선언한 규칙으로 0.3 선택

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">놓치지 않는 것이 우선</td>
<td style="text-align: left;">재현율 요구조건을 먼저 정함</td>
<td style="text-align: left;">암 · 사기</td>
</tr>
<tr>
<td style="text-align: left;">잘못 울리는 경보 비용이 큼</td>
<td style="text-align: left;">FP 비용 또는 정밀도를 함께 봄</td>
<td style="text-align: left;">스팸 · 알람</td>
</tr>
<tr>
<td style="text-align: left;">두 실수를 비슷하게</td>
<td style="text-align: left;">validation F1 최대를 후보로</td>
<td style="text-align: left;">이번 실습</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 실제 업무에서 F1 최대가 정답인 것은 아님 → FP · FN의 실제 비용으로 고르기도 함
> - 임곗값에는 모두에게 같은 정답이 없음 → 선택 규칙을 먼저 선언

> ✅ **한 줄 정리** · 임곗값을 낮추면 놓침은 줄고 헛발질은 늘며, 선택 규칙을 먼저 선언한 뒤 validation 표에서 고른다.

**딥러닝 코드에서는**

- `(torch.sigmoid(logit) >= thr)` — 기준 0.5는 기본값일 뿐, val에서 업무 기준으로

### 3-3. `A` test 최종 평가 1회 · `predict_proba(X_test)` <sub>원본 셀 35 · 36</sub>

> 🎯 **고른 기준으로 test를 처음 열면 validation과 비슷하게 나오는가?**

💡 **쉽게 말하면**

- 기준은 이미 확정 → test는 **선택된 기준 하나만** 평가.
- 결과를 보고 기준을 다시 바꾸지 않음.

**실행 흐름**

1. test 확률
2. 기준 0.3으로 0 / 1
3. 정밀도 · 재현율 · F1

**먼저 예측** · test F1은 validation F1 0.946보다 높을까 낮을까? — 답은 코드 실행 결과 아래에

In [10]:
proba_test_7 = threshold_pipe_7.predict_proba(X_final_test_7)[:, 1]  # test 처음 사용
y_pred_test_7 = (proba_test_7 >= selected_threshold_7).astype(int)   # 고른 기준 그대로
test_precision_7 = precision_score(y_final_test_7, y_pred_test_7, zero_division=0)  # 헛발질
test_recall_7 = recall_score(y_final_test_7, y_pred_test_7)          # 놓침
test_f1_7 = f1_score(y_final_test_7, y_pred_test_7)                  # 균형
print(f"선택된 threshold: {selected_threshold_7:.1f}")               # 0.3
print(f"FINAL TEST — Precision={test_precision_7:.3f}, "             # 보고할 숫자
      f"Recall={test_recall_7:.3f}, F1={test_f1_7:.3f}")

선택된 threshold: 0.3
FINAL TEST — Precision=0.919, Recall=0.944, F1=0.932


**결과** · 예측 정답은 조금 낮음 — 0.932

- `선택된 threshold: 0.3` — validation에서 정한 값
- `FINAL TEST — Precision=0.919, Recall=0.944, F1=0.932` — validation 0.946보다 조금 낮음 · 그대로 보고

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">validation F1 0.946</td>
<td style="text-align: left;">고를 때 쓴 점수 (낙관적)</td>
<td style="text-align: left;">선택</td>
</tr>
<tr>
<td style="text-align: left;">test F1 0.932</td>
<td style="text-align: left;">처음 보는 데이터</td>
<td style="text-align: left;"><strong>보고</strong></td>
</tr>
<tr>
<td style="text-align: left;">test를 보고 기준 재선택</td>
<td style="text-align: left;">test가 검증이 됨</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - test 자체를 보는 것이 금지가 아니라, test 결과를 선택의 피드백으로 쓰면 final test 역할을 잃음

> ✅ **한 줄 정리** · validation에서 고른 기준 하나로 test를 한 번 평가해 보고하며, 결과를 보고 기준을 바꾸지 않는다.

**딥러닝 코드에서는**

- `best.pt` + val에서 고른 기준 → test 1회 (논문 표의 숫자)

### 3-4. `B` seed를 바꿔 절차 반복 · `def run_threshold_experiment` <sub>원본 셀 37 ~ 41</sub>

> 🎯 **data split seed가 바뀌면 고른 임곗값과 test 결과는 어떻게 달라지는가?**

💡 **쉽게 말하면**

- 이 표는 **다시 고르기 위한 표가 아님** → 같은 절차를 여러 분할에서 반복한 독립 실험.
- 고정된 정답 기준은 없고, 중요한 것은 test를 보지 않고 고른 **절차**.

**실행 흐름**

1. seed마다 3분할 → train fit → validation에서 F1 최대 기준
2. 그 기준으로 test 1회
3. seed 1 · 7 · 42 비교

**먼저 예측** · seed마다 고른 기준이 같을까? — 답은 코드 실행 결과 아래에

In [11]:
def run_threshold_experiment(seed):                                   # 3장 절차 전체를 한 번
    Xd, Xt, yd, yt = train_test_split(                                # 개발 · test
        X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=seed)
    Xf, Xv, yf, yv = train_test_split(                                # train · validation
        Xd, yd, test_size=0.25, stratify=yd, random_state=seed)
    lr_s = LogisticRegression(max_iter=5000, C=0.1, random_state=42)  # 같은 약한 모델
    # 같은 구조
    pipe = Pipeline([("pca", PCA(n_components=10, random_state=42)), ("logistic", lr_s)])
    pv = pipe.fit(Xf, yf).predict_proba(Xv)[:, 1]                     # train fit · validation 확률
    f1s = [f1_score(yv, pv >= t) for t in thresholds_7]               # 후보마다 F1
    thr = float(thresholds_7[int(np.argmax(f1s))])                    # validation F1 최대
    yh = (pipe.predict_proba(Xt)[:, 1] >= thr).astype(int)            # 선택이 끝난 뒤 test 1회
    return seed, thr, precision_score(yt, yh), recall_score(yt, yh), f1_score(yt, yh)  # 한 행


seed_rows = [run_threshold_experiment(s) for s in [1, 7]]                     # 새 실험 2개
# seed 42는 3-3 재사용
seed_rows.append((42, selected_threshold_7, test_precision_7, test_recall_7, test_f1_7))
cols = ["seed", "선택 threshold", "test P", "test R", "test F1"]              # 열 이름
print(pd.DataFrame(seed_rows, columns=cols).round(3).to_string(index=False))  # 원본 셀 38의 표

 seed  선택 threshold  test P  test R  test F1
    1           0.7   1.000   0.833    0.909
    7           0.4   0.897   0.972    0.933
   42           0.3   0.919   0.944    0.932

**결과** · 예측 정답은 다름 — 0.7 · 0.4 · 0.3

- `1             0.7   1.000   0.833    0.909` — seed 1은 높은 기준 · 헛발질 0 · 놓침이 많음
- `7             0.4   0.897   0.972    0.933` — seed 7은 낮은 기준 · 놓침이 적음
- `42             0.3   0.919   0.944    0.932` — 3-3과 같은 행

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">같은 F1 근처</td>
<td style="text-align: left;">정밀도 · 재현율은 꽤 다름</td>
<td style="text-align: left;">모델의 행동은 지표 하나로 안 보임</td>
</tr>
<tr>
<td style="text-align: left;">임곗값</td>
<td style="text-align: left;">점수를 높이는 숫자가 아니라 어떤 실수를 감수할지 정하는 기준</td>
<td style="text-align: left;">업무 규칙으로</td>
</tr>
<tr>
<td style="text-align: left;">seed 반복 표</td>
<td style="text-align: left;">절차의 안정성 확인</td>
<td style="text-align: left;">다시 고르는 용도 아님</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - test 결과를 보고 threshold나 seed를 다시 고르지 않기
> - seed 42 행은 3-3에서 이미 test를 1회 평가했으므로 그 결과를 재사용 (test를 두 번 보지 않음)

> ✅ **한 줄 정리** · 데이터 분할이 바뀌면 고른 기준도 흔들리므로, 고정된 정답 기준보다 test를 보지 않고 고른 절차가 중요하다.

**딥러닝 코드에서는**

- seed 3개 × 같은 절차 → 평균 ± 표준편차로 보고

---
## 4. ROC · PR 곡선

> **이 장의 흐름**
>
> - validation 확률로 ROC 곡선 · AUC (전부 0 모델과 비교)
> - 그다음 PR 곡선 · AP와 기준선(양성 비율) → 두 곡선을 같은 근거의 두 관점으로

<img src="images/pfm4_curves.png" alt="validation ROC 곡선과 PR 곡선" width="912">

**파이프라인 위치** · ⑥ 평가 — 모든 기준에서의 요약

### 4-1. `A` ROC 곡선과 AUC · `roc_curve` · `roc_auc_score` <sub>원본 셀 42 ~ 46</sub>

> 🎯 **ROC-AUC는 무엇을 재고, 전부 0 모델은 몇이 나오는가?**

💡 **쉽게 말하면**

- 기준을 0 → 1로 움직이며 가로 **FPR**, 세로 **재현율 TPR**을 찍은 곡선.
- 왼쪽 위로 갈수록 좋음 · AUC 0.5 = 랜덤 · 1.0 = 완벽.

**실행 흐름**

1. validation 확률로 ROC 곡선
2. AUC 계산
3. 전부 0 확률과 비교

**먼저 예측** · 모든 확률이 0인 모델의 AUC는? — 답은 코드 실행 결과 아래에

In [12]:
from sklearn.metrics import roc_auc_score, roc_curve  # ROC

fpr_7, tpr_7, _ = roc_curve(y_val_7, proba_val_7)                       # 기준마다 (FPR, TPR)
auc_pipe_val_7 = roc_auc_score(y_val_7, proba_val_7)                    # 곡선 아래 면적
auc_dummy_val_7 = roc_auc_score(y_val_7, np.zeros(len(y_val_7)))        # 전부 0 확률
# 원본 셀 45
print(f"[validation ROC-AUC] clean Pipeline: {auc_pipe_val_7:.3f} | Dummy: {auc_dummy_val_7:.3f}")
ends = [(float(fpr_7[i]), float(tpr_7[i])) for i in (0, -1)]            # 곡선의 첫 점 · 끝 점
print("곡선 점 개수:", len(fpr_7), "· 시작", ends[0], "· 끝", ends[1])  # (0, 0) → (1, 1)

[validation ROC-AUC] clean Pipeline: 0.997 | Dummy: 0.500
곡선 점 개수: 12 · 시작 (0.0, 0.0) · 끝 (1.0, 1.0)


**결과** · 예측 정답은 0.500 — 구분 능력 없음

- `[validation ROC-AUC] clean Pipeline: 0.997 | Dummy: 0.500` — 약한 모델도 순서 매기기는 거의 완벽 · 전부 0은 랜덤과 같음
- `곡선 점 개수: 12 · 시작 (0.0, 0.0) · 끝 (1.0, 1.0)` — 점 12개 · 기준 ∞ → (0, 0), 기준 0 → (1, 1)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">FPR</td>
<td style="text-align: left;">FP ÷ (FP + TN) · 음성 중 잘못 울린 비율</td>
<td style="text-align: left;">가로축</td>
</tr>
<tr>
<td style="text-align: left;">TPR (= 재현율)</td>
<td style="text-align: left;">TP ÷ (TP + FN)</td>
<td style="text-align: left;">세로축</td>
</tr>
<tr>
<td style="text-align: left;">ROC-AUC</td>
<td style="text-align: left;">양성 · 음성 점수 분포가 얼마나 잘 갈리나</td>
<td style="text-align: left;">기준을 고르기 전 모델 비교 · 비교적 균형일 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - ROC-AUC 0.997이어도 기준을 잘못 고르면 F1은 낮을 수 있음 (3장)

> ✅ **한 줄 정리** · ROC-AUC는 모든 기준에서 양성과 음성을 얼마나 잘 구분하는지 요약하며, 0.5는 랜덤이다.

**딥러닝 코드에서는**

- 의료 · 이상 탐지 논문의 AUROC — 기준과 무관한 순위 성능

### 4-2. `A` PR 곡선과 AP · `precision_recall_curve` · `average_precision_score` <sub>원본 셀 47 ~ 51</sub>

> 🎯 **PR-AP는 무엇을 보고, 기준선은 어디인가?**

💡 **쉽게 말하면**

- 가로 재현율 · 세로 정밀도 → 오른쪽 위가 좋음.
- AP = Positive를 점수순으로 세웠을 때 **위쪽에 얼마나 모였나** (랭킹 품질).

**실행 흐름**

1. validation PR 곡선
2. AP 계산
3. 양성 비율(기준선)과 비교

**먼저 예측** · 랜덤 모델의 AP는 몇 근처일까? — 답은 코드 실행 결과 아래에

In [13]:
from sklearn.metrics import average_precision_score, precision_recall_curve  # PR

prec_7, rec_7, _ = precision_recall_curve(y_val_7, proba_val_7)  # 기준마다 (재현율, 정밀도)
ap_pipe_val_7 = average_precision_score(y_val_7, proba_val_7)    # PR 곡선 요약 (계단식 면적)
print(f"[validation AP] clean Pipeline: {ap_pipe_val_7:.3f}")    # 원본 셀 49
print(f"기준선(양성 비율): {y_val_7.mean():.2f}")                # 랜덤 모델의 AP
# 같은 확률 · 두 관점
print("ROC-AUC vs AP:", round(auc_pipe_val_7, 3), "vs", round(ap_pipe_val_7, 3))

[validation AP] clean Pipeline: 0.979
기준선(양성 비율): 0.10
ROC-AUC vs AP: 0.997 vs 0.979


**결과** · 예측 정답은 0.10 — 양성 비율

- `[validation AP] clean Pipeline: 0.979` — 높은 점수를 받은 것이 대부분 진짜 7
- `기준선(양성 비율): 0.10` — 랜덤이면 AP ≈ 0.10
- `ROC-AUC vs AP: 0.997 vs 0.979` — 같은 확률을 두 관점으로 · AP가 더 엄격

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">ROC 곡선</td>
<td style="text-align: left;">가로 FPR · 세로 TPR · 왼쪽 위</td>
<td style="text-align: left;">클래스가 비교적 균형일 때 전체 요약</td>
</tr>
<tr>
<td style="text-align: left;">PR 곡선</td>
<td style="text-align: left;">가로 재현율 · 세로 정밀도 · 오른쪽 위</td>
<td style="text-align: left;">불균형에서 더 직관적</td>
</tr>
<tr>
<td style="text-align: left;">둘 다</td>
<td style="text-align: left;">같은 validation 근거의 두 관점</td>
<td style="text-align: left;">Positive 정의 · 업무 비용에 맞춰 함께</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - AP는 PR 곡선 아래 면적을 계단식으로 근사한 값 (사다리꼴 면적과 조금 다를 수 있음)
> - 둘 중 하나를 자동 정답으로 고르지 않음

> ✅ **한 줄 정리** · PR-AP는 Positive가 점수 위쪽에 얼마나 모였는지 보는 지표이며, 기준선은 양성 비율이다.

**딥러닝 코드에서는**

- 객체 탐지 mAP = 클래스별 AP의 평균 (같은 PR 곡선)

---
## 5. 종합 실습 · 타이타닉에 같은 절차 · Dummy 0.7765와 비교

> **이 장의 흐름**
>
> - 1 ~ 4장을 한 번에 사용
> - 1-1과 같은 분할에서 train을 다시 train / validation으로 나누고, 기준을 validation에서 고른 뒤 test 1회 → Dummy와 지표별 비교

<img src="images/pfm4_practice.png" alt="타이타닉 test · Dummy vs 로지스틱 지표별 비교" width="620">

**파이프라인 위치** · ② 분할 → ③ 전처리(Pipeline) → ⑤ 기준 선택(validation) → ⑥ test 1회

> 🎯 **규칙 Dummy와 학습한 로지스틱을 정확도 · 정밀도 · 재현율 · F1 · AUC로 나란히 보면?**

💡 **쉽게 말하면**

- 정확도 하나가 아니라 **지표 다섯 개**로 비교.
- 기준은 validation에서, test는 한 번.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">1-1의 train_df를 train / validation으로</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">숫자 4개 특성 · Pipeline(Standard + 로지스틱)</td>
<td style="text-align: left;">파일별 3 · 5-1</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">규칙 선언 → validation F1 최대 기준</td>
<td style="text-align: left;">3-2</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">test 1회 · Dummy와 지표별 비교</td>
<td style="text-align: left;">1-1 · 2-2 · 4-1</td>
</tr>
</tbody>
</table>

**먼저 예측** · 로지스틱의 test 정확도는 Dummy 0.7765를 넘을까? — 답은 코드 실행 결과 아래에

In [14]:
# 1 · 2단계: train_df를 다시 나누고 숫자 특성 4개로 Pipeline
from sklearn.preprocessing import StandardScaler  # 단위 맞추기

FEATS = ["Pclass", "Age", "Fare", "is_male"]  # 숫자 특성 4개


def prep(df):          # 성별을 0 / 1로 · Age는 train 평균으로
    # 3-1 회수
    out = df.assign(is_male=(df["Sex"] == "male").astype(int)).fillna({"Age": age_fill_value})
    return out[FEATS]  # 특성만


fit_df, val_df = train_test_split(                # 1-1의 train을 다시 둘로
    train_df, test_size=0.25, stratify=train_df["Survived"], random_state=0)
# 전처리 + 모델
pipe_t = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])
pipe_t.fit(prep(fit_df), fit_df["Survived"])      # train에만 fit
p_val = pipe_t.predict_proba(prep(val_df))[:, 1]  # validation 생존 확률


# 3단계: 규칙(F1 최대)으로 기준 고르기
f1s = [f1_score(val_df["Survived"], p_val >= t) for t in thresholds_7]  # 3장과 같은 후보 9개
thr_t = float(thresholds_7[int(np.argmax(f1s))])                        # validation F1 최대
# 선택 끝
print(f"train {len(fit_df)} · validation {len(val_df)} · test {len(test_df)} · 고른 기준 {thr_t}")

train 534 · validation 178 · test 179 · 고른 기준 0.5


In [15]:
# 4단계: test 1회 · Dummy와 지표별 비교
p_test = pipe_t.predict_proba(prep(test_df))[:, 1]                         # test 처음 사용
pred_lr_t = (p_test >= thr_t).astype(int)                                  # 고른 기준 그대로
board = {}                                                                 # 모델별 지표
# 같은 지표로
for name, pred, score in [("Dummy", pred_dummy, pred_dummy), ("로지스틱", pred_lr_t, p_test)]:
    yt_ = y_test_titanic                                                   # 짧은 이름
    board[name] = [accuracy_score(yt_, pred), precision_score(yt_, pred),  # Dummy AUC는 0 / 1로
                   recall_score(yt_, pred), f1_score(yt_, pred), roc_auc_score(yt_, score)]
names = ["정확도", "정밀도", "재현율", "F1", "ROC-AUC"]                    # 행 이름
print(pd.DataFrame(board, index=names).round(4).to_string())               # 비교표

          Dummy    로지스틱
정확도      0.7765  0.7765
정밀도      0.7302  0.7231
재현율      0.6667  0.6812
F1       0.6970  0.7015
ROC-AUC  0.7561  0.8252


**결과** · 예측 정답은 못 넘음 — 정확도는 0.7765로 똑같음 · 차이는 AUC에서 보임

- `train 534 · validation 178 · test 179 · 고른 기준 0.5` — 기준은 validation에서 확정 (이번엔 0.5)
- `정확도      0.7765  0.7765` — 정확도는 규칙 한 줄과 똑같음
- `재현율      0.6667  0.6812` — 생존자를 1명 더 찾음 · 정밀도는 조금 낮음
- `ROC-AUC  0.7561  0.8252` — 순위 능력은 0.07 차이 → 정확도로는 안 보이던 차이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Dummy (남자면 사망)</td>
<td style="text-align: left;">규칙 한 줄 · 확률이 없음</td>
<td style="text-align: left;">넘어야 할 기준선</td>
</tr>
<tr>
<td style="text-align: left;">로지스틱 + 기준 선택</td>
<td style="text-align: left;">확률 → validation에서 기준</td>
<td style="text-align: left;">기준선을 지표별로 넘는지 확인</td>
</tr>
<tr>
<td style="text-align: left;">정확도만 비교</td>
<td style="text-align: left;">똑같음 → '학습해도 소용없네'로 끝남</td>
<td style="text-align: left;">쓰지 않기 · 지표 여러 개로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 숫자 4개 특성으로는 규칙과 정확도가 같음 → 성별 외 정보가 아직 부족 (특성 · 인코딩은 파일별 5)
> - 특성을 더 넣는 개선은 파일별 5(4-1)에서 · 이 test를 보고 기준을 바꾸지 않기

> ✅ **한 줄 정리** · 규칙 기준선과 학습한 모델은 정확도 · 정밀도 · 재현율 · F1 · AUC를 나란히 놓고 비교해야 차이가 보인다.

**딥러닝 코드에서는**

- 학습 로그에 accuracy · F1 · AUROC를 함께 기록해 어떤 지표가 움직이는지 보기

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. 정확도의 함정</td>
<td style="text-align: left;">규칙 한 줄 0.7765 · 전부 0 모델 0.900인데 재현율 0</td>
<td style="text-align: left;">1-1 · 1-3</td>
</tr>
<tr>
<td style="text-align: left;">2. 혼동행렬</td>
<td style="text-align: left;">Positive = 7 · TN 324 · FP 0 · FN 36 · TP 0</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">3. 지표 계산</td>
<td style="text-align: left;">P 34 ÷ 34 = 1 · R 34 ÷ 36 = 0.9444 · Balanced 0.9722 = Macro Recall</td>
<td style="text-align: left;">2-2 · 2-3</td>
</tr>
<tr>
<td style="text-align: left;">4. 임곗값</td>
<td style="text-align: left;">규칙 먼저 → validation 0.3 → test F1 0.932 · seed마다 0.7 · 0.4 · 0.3</td>
<td style="text-align: left;">3-1 ~ 3-4</td>
</tr>
<tr>
<td style="text-align: left;">5. ROC · PR</td>
<td style="text-align: left;">AUC 0.997 (Dummy 0.5) · AP 0.979 (기준선 0.10)</td>
<td style="text-align: left;">4-1 · 4-2</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">기준선</td>
<td style="text-align: left;">규칙 · 전부 0 정확도부터</td>
<td style="text-align: left;">첫 epoch 전 기준선</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Positive</td>
<td style="text-align: left;">찾고 싶은 대상을 먼저 적기</td>
<td style="text-align: left;">클래스 정의</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">혼동행렬</td>
<td style="text-align: left;">행 = 실제 · 열 = 예측 · TN FP / FN TP</td>
<td style="text-align: left;"><code>ConfusionMatrix</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">정밀도 · 재현율</td>
<td style="text-align: left;">헛발질 · 놓침</td>
<td style="text-align: left;">epoch마다 기록</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">F1</td>
<td style="text-align: left;">조화평균 · 하나가 낮으면 깎임</td>
<td style="text-align: left;">macro-F1</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">임곗값</td>
<td style="text-align: left;">규칙 먼저 → validation → test 1회</td>
<td style="text-align: left;"><code>sigmoid(logit) &gt;= thr</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">ROC-AUC</td>
<td style="text-align: left;">모든 기준의 구분 능력 · 0.5 = 랜덤</td>
<td style="text-align: left;">AUROC</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">PR-AP</td>
<td style="text-align: left;">Positive 랭킹 품질 · 기준선 = 양성 비율</td>
<td style="text-align: left;">mAP</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">Balanced · Macro · Weighted</td>
<td style="text-align: left;">재현율 평균 · 단순 평균 · 표본 가중</td>
<td style="text-align: left;">리더보드 지표</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">zero_division</td>
<td style="text-align: left;">양성 예측 0개일 때 값 정하기</td>
<td style="text-align: left;">경고 방지</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">'남자면 사망' 규칙의 정확도는?</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">digits test에서 7의 비율은?</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">전부 0 모델의 정확도 · 재현율은?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">혼동행렬의 행과 열은?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">기준선 로지스틱의 정밀도 · 재현율 계산은?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">Balanced Accuracy와 Macro Recall이 같은 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">임곗값을 고를 때 test를 쓰면 안 되는 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">임곗값을 낮추면?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">test 결과가 validation보다 낮으면?</td>
</tr>
<tr>
<td style="text-align: left;">3-4</td>
<td style="text-align: left;">seed마다 기준이 다르면?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">ROC-AUC 0.5의 뜻은?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">PR-AP의 기준선은?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. 3-2의 `thr_table_7`에서 재현율이 0.9 이상인 기준 중 정밀도가 가장 높은 기준을 출력
2. 그 기준의 validation F1을 소수 셋째 자리로 출력

In [16]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-2에서 Positive를 '7이 아닌 것'으로 바꾸면 정밀도 · 재현율은?
2. 3-2의 규칙을 '정밀도 0.95 이상 중 재현율 최대'로 바꾸면 기준은?
3. 4-2에서 모든 확률을 무작위(`np.random.RandomState(0).rand`)로 바꾸면 AUC · AP는?

In [17]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">0.7765 · 학습 없는 규칙도 높게 나옴</td>
</tr>
<tr>
<td style="text-align: left;">1-2</td>
<td style="text-align: left;">0.1 (360장 중 36장)</td>
</tr>
<tr>
<td style="text-align: left;">1-3</td>
<td style="text-align: left;">0.900 · 0.0</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">행 = 실제, 열 = 예측 · TN FP / FN TP</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">34 ÷ (34 + 0) = 1.000 · 34 ÷ (34 + 2) = 0.9444</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">둘 다 클래스별 재현율의 단순 평균 (정의상 같음)</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">선택에 쓴 데이터는 처음 보는 시험이 아니게 됨</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">재현율 ↑ (놓침 ↓) · 정밀도 ↓ (헛발질 ↑)</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">그대로 보고 · 기준을 다시 고르지 않음</td>
</tr>
<tr>
<td style="text-align: left;">3-4</td>
<td style="text-align: left;">정답 기준은 없음 · test를 보지 않고 고른 절차가 중요</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">랜덤 · 구분 능력 없음 (전부 0 확률)</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">양성 비율 (digits validation 0.10)</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
ok = thr_table_7[thr_table_7["recall"] >= 0.9]
pick = ok.loc[ok["precision"].idxmax()]
print(pick["threshold"])  # 0.5
print(round(pick["f1"], 3))  # 0.93
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">정밀도 0.9939 · 재현율 1.0000 — 7이 아닌 324장을 전부 찾고, 그렇게 예측한 326장 중 2장이 7</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">0.7 (정밀도 0.968 · 재현율 0.833) — 규칙이 바뀌면 같은 표에서 다른 기준</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">AUC 0.555 · AP 0.159 — 각각의 기준선(0.5 · 양성 비율 0.10) 근처 · 360장이라 우연히 조금 위</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) 정확도 0.9면 좋은 모델이라고 답함</td>
<td style="text-align: left;">양성 비율 · 기준선을 안 봄</td>
<td style="text-align: left;">혼동행렬 · 재현율부터</td>
<td style="text-align: left;">전부 0으로 찍으면 몇 점인가?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 정확도만으로 분류 모델을 평가하면 안 되는 이유는?**
> - 정확도는 맞힌 개수만 세기 때문에 어떤 실수를 했는지 알 수 없습니다.
> - digits에서 7이 10%일 때 전부 0으로 찍어도 정확도가 0.900이지만 재현율은 0입니다.
> - 그래서 혼동행렬로 실수를 나누고 업무 비용에 맞춰 정밀도와 재현율, F1을 함께 봅니다.

**Q2. 임곗값은 어떻게 정하나요?**
> - 확률을 0과 1로 바꾸는 기준은 0.5가 정답이 아닙니다.
> - F1 최대나 재현율 0.9 이상처럼 선택 규칙을 먼저 정하고, validation에서 그 규칙에 맞는 기준을 고른 뒤 test는 한 번만 평가합니다.
> - test 결과를 보고 다시 고르면 test가 검증 데이터가 됩니다.

**Q3. ROC-AUC와 PR-AP는 어떻게 다른가요?**
> - ROC-AUC는 모든 기준에서 양성과 음성의 점수가 얼마나 잘 갈리는지 보는 지표로 0.5가 랜덤입니다.
> - PR-AP는 양성이 점수 위쪽에 얼마나 모였는지를 보며 기준선이 양성 비율이라 불균형 데이터에서 더 엄격합니다.
> - 둘 다 같은 확률의 두 관점이라 함께 봅니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>BaseEstimator</code></td>
<td style="text-align: left;">sklearn.base</td>
<td style="text-align: left;">규칙 분류기 틀</td>
<td style="text-align: left;">1-1 <code>class MyDummyClassifier(BaseEstimator)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>load_digits</code></td>
<td style="text-align: left;">sklearn.datasets</td>
<td style="text-align: left;">8 × 8 숫자 1,797장</td>
<td style="text-align: left;">1-2 <code>digits = load_digits()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>precision_score</code> · <code>recall_score</code> · <code>f1_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">정밀도 · 재현율 · F1</td>
<td style="text-align: left;">1-3 <code>precision_score(…, zero_division=0)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>confusion_matrix(labels=)</code> · <code>ravel()</code></td>
<td style="text-align: left;">sklearn.metrics · ndarray</td>
<td style="text-align: left;">4칸</td>
<td style="text-align: left;">2-1 <code>confusion_matrix(y_test_7, fakepred_7, labels=[0, 1])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>classification_report</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">클래스별 표</td>
<td style="text-align: left;">2-1 <code>classification_report(…, digits=3)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>balanced_accuracy_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">클래스별 재현율 평균</td>
<td style="text-align: left;">2-3 <code>balanced_accuracy_score(y_test_7, y_pred_lr_7)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>precision_recall_fscore_support(average=)</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">macro · weighted</td>
<td style="text-align: left;">2-3 <code>average="macro"</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>PCA</code> · <code>Pipeline</code></td>
<td style="text-align: left;">sklearn</td>
<td style="text-align: left;">일부러 약한 모델 · train에만 fit</td>
<td style="text-align: left;">3-1 <code>Pipeline([("pca", PCA(…)), …])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>predict_proba()[:, 1]</code> · <code>&gt;= thr</code></td>
<td style="text-align: left;">분류 모델</td>
<td style="text-align: left;">확률 → 0 / 1</td>
<td style="text-align: left;">3-2 <code>(proba_val_7 &gt;= thr).astype(int)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>roc_curve</code> · <code>roc_auc_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">ROC · AUC</td>
<td style="text-align: left;">4-1 <code>roc_auc_score(y_val_7, proba_val_7)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>precision_recall_curve</code> · <code>average_precision_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">PR · AP</td>
<td style="text-align: left;">4-2 <code>average_precision_score(y_val_7, …)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">기준선부터 계산</td>
<td style="text-align: left;">정확도 하나로 결론</td>
</tr>
<tr>
<td style="text-align: left;">Positive를 먼저 적기</td>
<td style="text-align: left;">Positive를 정하지 않고 지표 읽기</td>
</tr>
<tr>
<td style="text-align: left;">선택 규칙을 표보다 먼저</td>
<td style="text-align: left;">표를 보고 규칙 정하기</td>
</tr>
<tr>
<td style="text-align: left;">기준은 validation에서</td>
<td style="text-align: left;">test를 보고 기준 바꾸기</td>
</tr>
<tr>
<td style="text-align: left;">ROC · PR을 함께</td>
<td style="text-align: left;">AUC가 높으니 F1도 높다고 가정</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정밀도 vs 재현율</td>
<td style="text-align: left;">헛발질 vs 놓침</td>
<td style="text-align: left;">비싼 실수 쪽을 우선</td>
</tr>
<tr>
<td style="text-align: left;">FP vs FN</td>
<td style="text-align: left;">음성을 양성으로 vs 양성을 음성으로</td>
<td style="text-align: left;">업무 비용 먼저 적기</td>
</tr>
<tr>
<td style="text-align: left;">Macro vs Weighted</td>
<td style="text-align: left;">단순 평균 vs 표본 가중</td>
<td style="text-align: left;">소수 클래스가 중요하면 macro</td>
</tr>
<tr>
<td style="text-align: left;">validation vs test</td>
<td style="text-align: left;">고르기 vs 보고</td>
<td style="text-align: left;">기준은 validation</td>
</tr>
<tr>
<td style="text-align: left;">ROC-AUC vs PR-AP</td>
<td style="text-align: left;">0.5 기준 vs 양성 비율 기준</td>
<td style="text-align: left;">불균형이면 AP도 함께</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 · 2-2 공식을 4칸 숫자로 손 계산</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">5장 종합 실습을 빈 셀에 처음부터 · 규칙을 '재현율 0.8 이상'으로 바꿔서</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 파일별 5(로지스틱) 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 셀 55개를 원본 Part 순서대로 1 ~ 4장 · 12개 섹션으로 나눔
- 원본 plotly 그림 6개(분포 · 혼동행렬 · 임곗값 · ROC · AUC 막대 · PR · 비교 표)는 같은 숫자로 장 그림 · 결과 줄로 바꿈
- 원본 설명 스크린샷 9장(Accuracy는 전체 평균 · Recall · Precision · F1 · ROC · PR)은 💡 · 비교 표의 글로
- 원본 assert(값 고정 검사)는 결과 줄 대조로 대신함 · 5장 종합 실습은 원본 마지막 예고(Dummy 0.7765와 비교)를 미리 해 봄

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">셀 0 ~ 2</td>
<td style="text-align: left;">목표 · 왜 정확도만 보면 안 되나 · 지표 요약</td>
<td style="text-align: left;">0. 원본 목표 · 1-3 비교 표</td>
</tr>
<tr>
<td style="text-align: left;">셀 3</td>
<td style="text-align: left;">버전 확인</td>
<td style="text-align: left;">0. 준비 코드</td>
</tr>
<tr>
<td style="text-align: left;">셀 4 ~ 8</td>
<td style="text-align: left;">타이타닉 Dummy</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 9 ~ 12</td>
<td style="text-align: left;">불균형 · digits 7 vs 나머지</td>
<td style="text-align: left;">1-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 13 ~ 15</td>
<td style="text-align: left;">All-0 결과</td>
<td style="text-align: left;">1-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 16 ~ 20</td>
<td style="text-align: left;">Positive 정하기 · 혼동행렬</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 21 ~ 25</td>
<td style="text-align: left;">P · R · F1 · 업무별 중요도 · 기준선</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 26 · 27</td>
<td style="text-align: left;">Balanced · Macro · Weighted</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 28 ~ 32</td>
<td style="text-align: left;">임곗값 · 역할 회수 · 3분할 · 약한 모델</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 33 · 34</td>
<td style="text-align: left;">선택 규칙 · validation 표</td>
<td style="text-align: left;">3-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 35 · 36</td>
<td style="text-align: left;">test 최종 1회</td>
<td style="text-align: left;">3-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 37 ~ 41</td>
<td style="text-align: left;">seed 반복 · Reveal · 그림 · 선택 가이드</td>
<td style="text-align: left;">3-4 · 3장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 42 ~ 46</td>
<td style="text-align: left;">ROC · AUC</td>
<td style="text-align: left;">4-1 · 4장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 47 ~ 51</td>
<td style="text-align: left;">PR · AP · ROC vs PR</td>
<td style="text-align: left;">4-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 52 ~ 54</td>
<td style="text-align: left;">오늘의 정리 · Ending · 다음 시간 예고</td>
<td style="text-align: left;">그래서 오늘 · 5. 종합 실습</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 5장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [18]:
from sklearn.base import BaseEstimator    # fit · predict 틀만 빌림
from sklearn.datasets import load_digits  # 8 × 8 손글씨 숫자


# ---------- 1-1 ----------

CSV = "../03_전처리_평가지표_Titanic/train.csv"                     # 원본 폴더의 Kaggle Titanic
titanic_df = pd.read_csv(CSV)                                       # 891명
train_df, test_df = train_test_split(titanic_df, test_size=0.2,     # 원본과 같은 분할
                                     stratify=titanic_df["Survived"], random_state=0)
age_fill_value = train_df["Age"].mean()  # 결측 대체값은 train에서만 (3-1 누수 회수)
print(f"Age 결측 대체값(train 평균만 사용): {age_fill_value:.4f}")  # 원본과 같은 값
# test는 transform만
X_test_titanic = test_df.drop("Survived", axis=1).fillna({"Age": age_fill_value})
y_test_titanic = test_df["Survived"]                                # 정답 (1 = 생존)


class MyDummyClassifier(BaseEstimator):  # 학습 없는 규칙
    def fit(self, X, y=None):            # 아무것도 배우지 않음
        return self                      # 그대로 돌려줌

    def predict(self, X):                                                 # 남자면 0, 아니면 1
        return np.where(X["Sex"].to_numpy() == "male", 0, 1).astype(int)  # 성별만 봄


pred_dummy = MyDummyClassifier().fit(None).predict(X_test_titanic)  # 규칙으로 예측
print("Dummy Classifier 정확도:", round(accuracy_score(y_test_titanic, pred_dummy), 4))  # 0.7765
print("test 생존 비율:", round(y_test_titanic.mean(), 4))           # 찍기 기준선 비교용


# ---------- 1-2 ----------

digits = load_digits()                                                # 1797장
X_all_7 = digits.data                                                 # 픽셀 64개
y_all_7 = (digits.target == 7).astype(int)                            # 7이면 1, 아니면 0
X_train_7, X_test_7, y_train_7, y_test_7 = train_test_split(          # 원본과 같은 분할
    X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=11)
# 0.9 · 0.1
print("테스트셋 클래스 비율:", pd.Series(y_test_7).value_counts(normalize=True).round(3).to_dict())
print("test 장수:", len(y_test_7), "· 그중 7:", int(y_test_7.sum()))  # 360장 중 36장


# ---------- 1-3 ----------
class FakeClassifier(BaseEstimator):  # 항상 0
    def fit(self, X, y=None):         # 배우지 않음
        return self                   # 그대로

    def predict(self, X):                   # 전부 0
        return np.zeros(len(X), dtype=int)  # '7 아님'만


fakepred_7 = FakeClassifier().fit(X_train_7, y_train_7).predict(X_test_7)  # 가짜 예측
print(f"Accuracy: {accuracy_score(y_test_7, fakepred_7):.3f}")             # 전체 성적표
# 7이라고 한 것 중 진짜 7
print("Precision:", precision_score(y_test_7, fakepred_7, zero_division=0))
print("Recall   :", recall_score(y_test_7, fakepred_7, zero_division=0))   # 진짜 7 중 찾은 것

Age 결측 대체값(train 평균만 사용): 29.4698
Dummy Classifier 정확도: 0.7765
test 생존 비율: 0.3855


테스트셋 클래스 비율: {0: 0.9, 1: 0.1}
test 장수: 360 · 그중 7: 36
Accuracy: 0.900
Precision: 0.0
Recall   : 0.0


In [19]:
from sklearn.metrics import classification_report, confusion_matrix  # 4칸 · 클래스별 표
from sklearn.linear_model import LogisticRegression                  # 기준선 모델
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support  # 평균 방식


# ---------- 2-1 ----------

cm_7 = confusion_matrix(y_test_7, fakepred_7, labels=[0, 1])  # 행 = 실제, 열 = 예측
print("TN, FP, FN, TP:", *cm_7.ravel())                       # 원본 셀 19
print(classification_report(y_test_7, fakepred_7, digits=3, zero_division=0))  # 클래스별 지표


# ---------- 2-2 ----------

lr_7 = LogisticRegression(max_iter=5000, random_state=42).fit(X_train_7, y_train_7)  # 원본 설정
y_pred_lr_7 = lr_7.predict(X_test_7)                                             # 기본 기준 0.5
tn, fp, fn, tp = confusion_matrix(y_test_7, y_pred_lr_7, labels=[0, 1]).ravel()  # 4칸
print(f"Logistic Accuracy: {accuracy_score(y_test_7, y_pred_lr_7):.4f}")         # 원본 셀 25
print("Baseline TN, FP, FN, TP:", tn, fp, fn, tp)                                # 칸 숫자
p_hand, r_hand = tp / (tp + fp), tp / (tp + fn)                                  # 공식 그대로
f1_hand = 2 * p_hand * r_hand / (p_hand + r_hand)                                # 조화평균
print(f"손 계산 P {p_hand:.4f} · R {r_hand:.4f} · F1 {f1_hand:.4f}")             # 직접
p_fn = precision_score(y_test_7, y_pred_lr_7)                                    # 함수로 정밀도
r_fn = recall_score(y_test_7, y_pred_lr_7)                                       # 함수로 재현율
f_fn = f1_score(y_test_7, y_pred_lr_7)                                           # 함수로 F1
print(f"함수   P {p_fn:.4f} · R {r_fn:.4f} · F1 {f_fn:.4f}")                     # sklearn과 같은지


# ---------- 2-3 ----------

bal = balanced_accuracy_score(y_test_7, y_pred_lr_7)                       # 클래스별 재현율 평균
pm, rm, fm, _ = precision_recall_fscore_support(y_test_7, y_pred_lr_7, average="macro")  # 단순 평균
# 표본 수 가중
pw, rw, fw, _ = precision_recall_fscore_support(y_test_7, y_pred_lr_7, average="weighted")
print(f"Balanced Accuracy: {bal:.4f}")                                     # 원본 셀 26
print(f"Macro     P:{pm:.4f} R:{rm:.4f} F1:{fm:.4f}")                      # 소수 클래스도 같은 비중
print(f"Weighted  P:{pw:.4f} R:{rw:.4f} F1:{fw:.4f}")                      # 다수 클래스 쪽
r0, r1 = tn / (tn + fp), tp / (tp + fn)                                    # 클래스 0 · 1의 재현율
print(f"(재현율0 {r0:.4f} + 재현율1 {r1:.4f}) / 2 = {(r0 + r1) / 2:.4f}")  # Balanced와 같은지

TN, FP, FN, TP: 324 0 36 0
              precision    recall  f1-score   support

           0      0.900     1.000     0.947       324
           1      0.000     0.000     0.000        36

    accuracy                          0.900       360
   macro avg      0.450     0.500     0.474       360
weighted avg      0.810     0.900     0.853       360

Logistic Accuracy: 0.9944
Baseline TN, FP, FN, TP: 324 0 2 34
손 계산 P 1.0000 · R 0.9444 · F1 0.9714
함수   P 1.0000 · R 0.9444 · F1 0.9714
Balanced Accuracy: 0.9722
Macro     P:0.9969 R:0.9722 F1:0.9842
Weighted  P:0.9945 R:0.9944 F1:0.9944
(재현율0 1.0000 + 재현율1 0.9444) / 2 = 0.9722


In [20]:
from sklearn.decomposition import PCA  # 정보를 10개로 줄임
from sklearn.pipeline import Pipeline  # train에만 fit


# ---------- 3-1 ----------

X_dev, X_final_test_7, y_dev, y_final_test_7 = train_test_split(       # 3장 전용 분할 (seed 42)
    X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=42)
X_fit_7, X_val_7, y_fit_7, y_val_7 = train_test_split(                 # 개발 데이터를 다시 나눔
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print("train      :", X_fit_7.shape, "Positive=", int(y_fit_7.sum()))  # 학습용
print("validation :", X_val_7.shape, "Positive=", int(y_val_7.sum()))  # 기준 선택용
n_pos_test = int(y_final_test_7.sum())                                 # test의 7 개수
# 봉인
print("test       :", X_final_test_7.shape, "Positive=", n_pos_test, "(아직 선택에 사용하지 않음)")
weak_lr = LogisticRegression(max_iter=5000, C=0.1, random_state=42)    # 규제를 준 로지스틱
threshold_pipe_7 = Pipeline([("pca", PCA(n_components=10, random_state=42)),  # 일부러 약하게
                             ("logistic", weak_lr)])
threshold_pipe_7.fit(X_fit_7, y_fit_7)                                 # PCA까지 train에만 fit
proba_val_7 = threshold_pipe_7.predict_proba(X_val_7)[:, 1]            # 7일 확률
print("validation probability 개수:", len(proba_val_7))                # 360개


# ---------- 3-2 ----------
thresholds_7 = np.round(np.arange(0.1, 1.0, 0.1), 1)                  # 후보 9개
rows = []                                                             # 기준마다 지표
for thr in thresholds_7:                                              # 기준을 하나씩
    y_hat = (proba_val_7 >= thr).astype(int)                          # 확률 ≥ 기준 → 7
    rows.append({"threshold": thr,                                    # 네 지표
                 "precision": precision_score(y_val_7, y_hat, zero_division=0),
                 "recall": recall_score(y_val_7, y_hat), "f1": f1_score(y_val_7, y_hat),
                 "accuracy": accuracy_score(y_val_7, y_hat)})
thr_table_7 = pd.DataFrame(rows)                                      # 9행 표
print(thr_table_7.round(3).to_string(index=False))                    # 원본 셀 34의 표
best_row_7 = thr_table_7.loc[thr_table_7["f1"].idxmax()]              # 미리 선언한 규칙
selected_threshold_7 = float(best_row_7["threshold"])                 # 고른 기준
b = best_row_7                                                        # 짧은 이름
print(f"[validation F1 최대] threshold={selected_threshold_7:.1f}, "  # 원본 형식
      f"P={b['precision']:.3f}, R={b['recall']:.3f}, F1={b['f1']:.3f}")


# ---------- 3-3 ----------
proba_test_7 = threshold_pipe_7.predict_proba(X_final_test_7)[:, 1]  # test 처음 사용
y_pred_test_7 = (proba_test_7 >= selected_threshold_7).astype(int)   # 고른 기준 그대로
test_precision_7 = precision_score(y_final_test_7, y_pred_test_7, zero_division=0)  # 헛발질
test_recall_7 = recall_score(y_final_test_7, y_pred_test_7)          # 놓침
test_f1_7 = f1_score(y_final_test_7, y_pred_test_7)                  # 균형
print(f"선택된 threshold: {selected_threshold_7:.1f}")               # 0.3
print(f"FINAL TEST — Precision={test_precision_7:.3f}, "             # 보고할 숫자
      f"Recall={test_recall_7:.3f}, F1={test_f1_7:.3f}")

train      : (1077, 64) Positive= 107
validation : (360, 64) Positive= 36
test       : (360, 64) Positive= 36 (아직 선택에 사용하지 않음)
validation probability 개수: 360


 threshold  precision  recall    f1  accuracy
       0.1      0.854   0.972 0.909     0.981
       0.2      0.897   0.972 0.933     0.986
       0.3      0.921   0.972 0.946     0.989
       0.4      0.919   0.944 0.932     0.986
       0.5      0.943   0.917 0.930     0.986
       0.6      0.939   0.861 0.899     0.981
       0.7      0.968   0.833 0.896     0.981
       0.8      0.964   0.750 0.844     0.972
       0.9      1.000   0.694 0.820     0.969
[validation F1 최대] threshold=0.3, P=0.921, R=0.972, F1=0.946
선택된 threshold: 0.3
FINAL TEST — Precision=0.919, Recall=0.944, F1=0.932


In [21]:
# ---------- 3-4 ----------
def run_threshold_experiment(seed):                                   # 3장 절차 전체를 한 번
    Xd, Xt, yd, yt = train_test_split(                                # 개발 · test
        X_all_7, y_all_7, test_size=0.2, stratify=y_all_7, random_state=seed)
    Xf, Xv, yf, yv = train_test_split(                                # train · validation
        Xd, yd, test_size=0.25, stratify=yd, random_state=seed)
    lr_s = LogisticRegression(max_iter=5000, C=0.1, random_state=42)  # 같은 약한 모델
    # 같은 구조
    pipe = Pipeline([("pca", PCA(n_components=10, random_state=42)), ("logistic", lr_s)])
    pv = pipe.fit(Xf, yf).predict_proba(Xv)[:, 1]                     # train fit · validation 확률
    f1s = [f1_score(yv, pv >= t) for t in thresholds_7]               # 후보마다 F1
    thr = float(thresholds_7[int(np.argmax(f1s))])                    # validation F1 최대
    yh = (pipe.predict_proba(Xt)[:, 1] >= thr).astype(int)            # 선택이 끝난 뒤 test 1회
    return seed, thr, precision_score(yt, yh), recall_score(yt, yh), f1_score(yt, yh)  # 한 행


seed_rows = [run_threshold_experiment(s) for s in [1, 7]]                     # 새 실험 2개
# seed 42는 3-3 재사용
seed_rows.append((42, selected_threshold_7, test_precision_7, test_recall_7, test_f1_7))
cols = ["seed", "선택 threshold", "test P", "test R", "test F1"]              # 열 이름
print(pd.DataFrame(seed_rows, columns=cols).round(3).to_string(index=False))  # 원본 셀 38의 표

 seed  선택 threshold  test P  test R  test F1
    1           0.7   1.000   0.833    0.909
    7           0.4   0.897   0.972    0.933
   42           0.3   0.919   0.944    0.932

In [22]:
from sklearn.metrics import roc_auc_score, roc_curve                         # ROC
from sklearn.metrics import average_precision_score, precision_recall_curve  # PR


# ---------- 4-1 ----------

fpr_7, tpr_7, _ = roc_curve(y_val_7, proba_val_7)                       # 기준마다 (FPR, TPR)
auc_pipe_val_7 = roc_auc_score(y_val_7, proba_val_7)                    # 곡선 아래 면적
auc_dummy_val_7 = roc_auc_score(y_val_7, np.zeros(len(y_val_7)))        # 전부 0 확률
# 원본 셀 45
print(f"[validation ROC-AUC] clean Pipeline: {auc_pipe_val_7:.3f} | Dummy: {auc_dummy_val_7:.3f}")
ends = [(float(fpr_7[i]), float(tpr_7[i])) for i in (0, -1)]            # 곡선의 첫 점 · 끝 점
print("곡선 점 개수:", len(fpr_7), "· 시작", ends[0], "· 끝", ends[1])  # (0, 0) → (1, 1)


# ---------- 4-2 ----------

prec_7, rec_7, _ = precision_recall_curve(y_val_7, proba_val_7)  # 기준마다 (재현율, 정밀도)
ap_pipe_val_7 = average_precision_score(y_val_7, proba_val_7)    # PR 곡선 요약 (계단식 면적)
print(f"[validation AP] clean Pipeline: {ap_pipe_val_7:.3f}")    # 원본 셀 49
print(f"기준선(양성 비율): {y_val_7.mean():.2f}")                # 랜덤 모델의 AP
# 같은 확률 · 두 관점
print("ROC-AUC vs AP:", round(auc_pipe_val_7, 3), "vs", round(ap_pipe_val_7, 3))

[validation ROC-AUC] clean Pipeline: 0.997 | Dummy: 0.500
곡선 점 개수: 12 · 시작 (0.0, 0.0) · 끝 (1.0, 1.0)
[validation AP] clean Pipeline: 0.979
기준선(양성 비율): 0.10
ROC-AUC vs AP: 0.997 vs 0.979


In [23]:
# ---------- 5. 종합 실습 ----------
from sklearn.preprocessing import StandardScaler  # 단위 맞추기


# 1 · 2단계: train_df를 다시 나누고 숫자 특성 4개로 Pipeline

FEATS = ["Pclass", "Age", "Fare", "is_male"]  # 숫자 특성 4개


def prep(df):          # 성별을 0 / 1로 · Age는 train 평균으로
    # 3-1 회수
    out = df.assign(is_male=(df["Sex"] == "male").astype(int)).fillna({"Age": age_fill_value})
    return out[FEATS]  # 특성만


fit_df, val_df = train_test_split(                # 1-1의 train을 다시 둘로
    train_df, test_size=0.25, stratify=train_df["Survived"], random_state=0)
# 전처리 + 모델
pipe_t = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])
pipe_t.fit(prep(fit_df), fit_df["Survived"])      # train에만 fit
p_val = pipe_t.predict_proba(prep(val_df))[:, 1]  # validation 생존 확률


# 3단계: 규칙(F1 최대)으로 기준 고르기
f1s = [f1_score(val_df["Survived"], p_val >= t) for t in thresholds_7]  # 3장과 같은 후보 9개
thr_t = float(thresholds_7[int(np.argmax(f1s))])                        # validation F1 최대
# 선택 끝
print(f"train {len(fit_df)} · validation {len(val_df)} · test {len(test_df)} · 고른 기준 {thr_t}")


# 4단계: test 1회 · Dummy와 지표별 비교
p_test = pipe_t.predict_proba(prep(test_df))[:, 1]                         # test 처음 사용
pred_lr_t = (p_test >= thr_t).astype(int)                                  # 고른 기준 그대로
board = {}                                                                 # 모델별 지표
# 같은 지표로
for name, pred, score in [("Dummy", pred_dummy, pred_dummy), ("로지스틱", pred_lr_t, p_test)]:
    yt_ = y_test_titanic                                                   # 짧은 이름
    board[name] = [accuracy_score(yt_, pred), precision_score(yt_, pred),  # Dummy AUC는 0 / 1로
                   recall_score(yt_, pred), f1_score(yt_, pred), roc_auc_score(yt_, score)]
names = ["정확도", "정밀도", "재현율", "F1", "ROC-AUC"]                    # 행 이름
print(pd.DataFrame(board, index=names).round(4).to_string())               # 비교표

train 534 · validation 178 · test 179 · 고른 기준 0.5
          Dummy    로지스틱
정확도      0.7765  0.7765
정밀도      0.7302  0.7231
재현율      0.6667  0.6812
F1       0.6970  0.7015
ROC-AUC  0.7561  0.8252
